In [ ]:
from typing_extensions import evaluate_forward_ref
from pandas.tseries.offsets import FY5253
import corner
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import matplotlib
from gwpy.timeseries import TimeSeries
import math
import bilby
from bilby.gw.conversion import generate_all_bbh_parameters
from bilby.core import utils
import numpy as np
from gwosc import datasets
import h5py
import pesummary
from pesummary.io import read
from pesummary.core.plots.plot import _make_comparison_corner_plot
import seaborn as sns
import pandas as pd
import itertools
import argparse
import warnings
import os
import json
import matplotlib.colors as mcolors
import configparser
import errno

def comparison_posterior_plot(pyring_label, bilby_label, default_plot_parameters=True, show_fig=True, plot_kde=True, bilby_label_v2=None):
    # 特定の警告を非表示にする
    warnings.filterwarnings("ignore", category=FutureWarning, 
                            module="seaborn._oldcore", lineno=1119)
    warnings.filterwarnings("ignore", category=FutureWarning, 
                            module="seaborn._oldcore", lineno=1498)
    
    """plot setting"""
    plt.style.use('seaborn-v0_8-colorblind')
    plt.style.use('seaborn-v0_8-whitegrid')
    first_color = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
    second_color = plt.rcParams['axes.prop_cycle'].by_key()['color'][1]
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    linestyles = ['--', '-', ':', '-.']

    """set labels"""
    latex_labels = {
        'mass_1_source': r'$m_{1,\mathrm{source}}\ [M_{\odot}]$',    
        'mass_2_source': r'$m_{2,\mathrm{source}}\ [M_{\odot}]$',    
        'mass_1': r'$m_{1}\ [M_{\odot}]$',    
        'mass_2': r'$m_{2}\ [M_{\odot}]$',    
        'chirp_mass_source': r'$\mathcal{M}_{\mathrm{source}}\ [M_{\odot}]$',
        'chirp_mass': r'$\mathcal{M}\ [M_{\odot}]$',
        'mass_ratio': r'$q$',
        'total_mass_source': r'$m_{\mathrm{total,source}}\ [M_{\odot}]$',
        'symmetric_mass_ratio': r'$\eta$',
        'chi_1': r'$\chi_{\rm 1}$',
        'chi_2': r'$\chi_{\rm 2}$',
        'chi_eff': r'$\chi_{\rm eff}$',
        'luminosity_distance': r'$d_L\ [{\rm Mpc}]$',
        'theta_jn': r'$\iota\ [{\rm rad}]$',
        'cos(iota)': r'$\cos \iota$',
        'sin(iota)': r'$\sin \iota$',
        'sin^2(iota)': r'$\sin^{2} \iota$',
        'geocent_time': r'$t_{c}$',
        'psi': r'$\psi$',
        'phase': r'$\phi$',
        'A_b1': r'$A_{b1}$',
        'A_b2': r'$A_{b2}$',
        'tilde_Ab1': r'$\tilde{A}_{b1}$',
        'tilde_Ab2': r'$\tilde{A}_{b2}$',
        'A1_real': r"$\mathrm{Re}[A_1] \times 10^{-20}$",
        'A1_imag': r"$\mathrm{Im}[A_1] \times 10^{-20}$",
        'alpha_real': r"$\mathrm{Re}[\alpha]$",
        'alpha_imag': r"$\mathrm{Im}[\alpha]$",
        'w1_real': r"$\mathrm{Re}[w_1]$",
        'w1_imag': r"$\mathrm{Im}[w_1]$",
        'w2_real': r"$\mathrm{Re}[w_2]$",
        'w2_imag': r"$\mathrm{Im}[w_2]$",
        'A': r'$A \times 10^{-17}$',
        'A1': r'$A_1 \times 10^{-19}$',
        'A2': r'$A_2 \times 10^{-19}$',
        'alpha': r'$\alpha \ [\mathrm{s}]$',
        'f': r'$f \ [\mathrm{Hz}]$',
        'f1': r'$f_1 \ [\mathrm{Hz}]$',
        'f2': r'$f_2 \ [\mathrm{Hz}]$',
        'tau': r'$\tau \ [\mathrm{s}]$',
        'tau1': r'$\tau_1 \ [\mathrm{s}]$',
        'tau2': r'$\tau_2 \ [\mathrm{s}]$',
        'phi1': r'$\phi_1 \ [\mathrm{rad}]$',
        'phi2': r'$\phi_2 \ [\mathrm{rad}]$',
        """for pyring"""
        'logA_t_0': r'$\log A_1$',
        'logA_t_1': r'$\log A_2$',
        'A_t_0': r'$A_1 \times 10^{-19}$',
        'A_t_1': r'$A_2 \times 10^{-19}$',
        'f_t_0': r'$f_1 \ [\mathrm{Hz}]$',
        'f_t_1': r'$f_2 \ [\mathrm{Hz}]$',
        'tau_t_0': r'$\tau_1 \ [\mathrm{s}]$',
        'tau_t_1': r'$\tau_2 \ [\mathrm{s}]$',
        'phi_t_0': r'$\phi_1 \ [\mathrm{rad}]$',
        'phi_t_1': r'$\phi_2 \ [\mathrm{rad}]$',
        'phiA': r'$\phi_A \ [\mathrm{rad}]$',
        'phialpha': r'$\phi_\alpha \ [\mathrm{rad}]$',
        'C': r'$C \times 10^{-21}$',
        'D': r'$D \times 10^{-17}$',
        'phiC': r'$\phi_C \ [\mathrm{rad}]$',
        'phiD': r'$\phi_D \ [\mathrm{rad}]$',
    }
    
    """set function of conversion"""
    def conversion_samples(sample_pandas):
        # INPUT
        # sample_dict: posteriors
        new_sample_list = []
        for _, sample in sample_pandas.iterrows():
            new_sample = generate_all_bbh_parameters(sample)
            new_sample_list.append(new_sample)
    
        new_sample_pandas = pd.DataFrame(new_sample_list)
        return new_sample_pandas
    
    """set data paths"""
    path_json_bilby = f'../run_bilby/outdirs/outdir_{bilby_label}/{bilby_label}_result.json'
    path_posterior_pyring = f'./outdirs/outdir_{pyring_label}/Nested_sampler/posterior.dat'
    path_config_pyring = f'./outdirs/outdir_{pyring_label}/{pyring_label}.ini'
    path_outdir = f'outdirs/outdir_{pyring_label}/'

    if bilby_label_v2 is not None:
        path_json_bilby_v2 = f'../run_bilby/outdirs/outdir_{bilby_label_v2}/{bilby_label_v2}_result.json'
    
    """read json file"""
    file_path_bilby = path_json_bilby
    data_bilby = read(file_path_bilby, package='core')
    with open(file_path_bilby, 'r') as f:
        others_data_bilby = json.load(f)
    posterior_samples_bilby = data_bilby.samples_dict.to_pandas()

    file_path_pyring = path_posterior_pyring
    df_pyring = pd.read_csv(file_path_pyring, sep='\s+')
    if '#' in df_pyring.columns:
        if df_pyring.iloc[:, -1].isna().all():
            real_columns = df_pyring.columns[1:].tolist()
            df_pyring = df_pyring.iloc[:, :-1]
            df_pyring.columns = real_columns
    
    if bilby_label_v2 is not None:
        file_path_bilby_v2 = path_json_bilby_v2
        data_bilby_v2 = read(file_path_bilby_v2, package='core')
        with open(file_path_bilby_v2, 'r') as f:
            others_data_bilby_v2 = json.load(f)
        posterior_samples_bilby_v2 = data_bilby_v2.samples_dict.to_pandas()

    """set injection value of version1"""
    injection_parameters_dict_bilby = data_bilby.injection_parameters
    if 'EPparam' in bilby_label:
        injection_parameters_dict_new = {}
        injection_parameters_dict_new['C'] = injection_parameters_dict_bilby['A'] * injection_parameters_dict_bilby['alpha'] * 1e4
        injection_parameters_dict_new['D'] = injection_parameters_dict_bilby['A']
        injection_parameters_dict_new['f'] = injection_parameters_dict_bilby['f1']
        injection_parameters_dict_new['tau'] = injection_parameters_dict_bilby['tau1']
        injection_parameters_dict_new['phiC'] = np.angle(-np.exp(1j*(injection_parameters_dict_bilby['phiA']+injection_parameters_dict_bilby['phialpha'])))
        injection_parameters_dict_new['phiD'] = injection_parameters_dict_bilby['phiA']
        injection_parameters_dict_bilby = injection_parameters_dict_new
    print(' injection_parameters bilby: {}'.format(injection_parameters_dict_bilby))

    if bilby_label_v2 is not None:
        injection_parameters_dict_bilby_v2 = data_bilby_v2.injection_parameters
        if 'EPparam' in bilby_label_v2:
            injection_parameters_dict_new = {}
            injection_parameters_dict_new['C'] = injection_parameters_dict_bilby_v2['A'] * injection_parameters_dict_bilby_v2['alpha'] * 1e4
            injection_parameters_dict_new['D'] = injection_parameters_dict_bilby_v2['A']
            injection_parameters_dict_new['f'] = injection_parameters_dict_bilby_v2['f1']
            injection_parameters_dict_new['tau'] = injection_parameters_dict_bilby_v2['tau1']
            injection_parameters_dict_new['phiC'] = np.angle(-np.exp(1j*(injection_parameters_dict_bilby_v2['phiA']+injection_parameters_dict_bilby_v2['phialpha'])))
            injection_parameters_dict_new['phiD'] = injection_parameters_dict_bilby_v2['phiA']
            injection_parameters_dict_bilby_v2 = injection_parameters_dict_new
        print(' injection_parameters bilby v2: {}'.format(injection_parameters_dict_bilby_v2))

    """set injection value of pyring"""
    config_ini = configparser.ConfigParser()
    config_ini.optionxform = str
    config_ini.read(path_config_pyring, encoding='utf-8')
    if not os.path.exists(path_config_pyring):
        raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), path_config_pyring)
    config_injection = config_ini['Injection']
    injection_parameters_dict_pyring = {}
    for key, val in config_injection.items():
        if 'A_' in key or key in ['A1', 'A2']:
            injection_parameters_dict_pyring[key] = float(val) * 1e19
            if key in df_pyring.columns:
                df_pyring[key] = df_pyring[key] * 1e19
        elif key == 'A':
            injection_parameters_dict_pyring[key] = float(val) * 1e17
            if key in df_pyring.columns:
                df_pyring[key] = df_pyring[key] * 1e17
        else:
            injection_parameters_dict_pyring[key] = float(val)
    if 'EPparam' in pyring_label:
        injection_parameters_dict_new = {}
        injection_parameters_dict_new['C'] = injection_parameters_dict_pyring['A'] * injection_parameters_dict_pyring['alpha'] * 1e4
        injection_parameters_dict_new['D'] = injection_parameters_dict_pyring['A']
        injection_parameters_dict_new['f'] = injection_parameters_dict_pyring['f1']
        injection_parameters_dict_new['tau'] = injection_parameters_dict_pyring['tau1']
        injection_parameters_dict_new['phiC'] = np.angle(-np.exp(1j*(injection_parameters_dict_pyring['phiA']+injection_parameters_dict_pyring['phialpha'])))
        injection_parameters_dict_new['phiD'] = injection_parameters_dict_pyring['phiA']
        injection_parameters_dict_pyring = injection_parameters_dict_new
    print(' injection_parameters pyring: {}'.format(injection_parameters_dict_pyring))
    print('')

    
    """set keys to plot for bilby"""
    if default_plot_parameters:
        keys_to_plot_bilby = others_data_bilby['search_parameter_keys']
    for key in keys_to_plot_bilby:
        print('plot key for bilby: {}'.format(key))
    
    if bilby_label_v2 is not None:
        if default_plot_parameters:
            keys_to_plot_bilby = others_data_bilby_v2['search_parameter_keys']
        for key in keys_to_plot_bilby:
            print('plot key for bilby v2: {}'.format(key))

    """fixed parameters"""
    if 'fixed_parameter_keys' in others_data_bilby.keys():
        fix_list_bilby = others_data_bilby['fixed_parameter_keys']
        for key in fix_list_bilby:
            print('not plot key for bilby: {}'.format(key))
    print('')

    if bilby_label_v2 is not None:
        if 'fixed_parameter_keys' in others_data_bilby_v2.keys():
            fix_list_bilby = others_data_bilby_v2['fixed_parameter_keys']
            for key in fix_list_bilby:
                print('not plot key for bilby v2: {}'.format(key))
        print('')


    """set keys to plot for pyring"""
    config_prior = config_ini['Priors']
    prior_parameters_dict = {key: value for key, value in config_prior.items()}

    keys_to_plot_pyring = []
    for key in prior_parameters_dict.keys():
        if 'max' in key:
            param_name = key.replace('-max', '')
            keys_to_plot_pyring.append(param_name)
            print('plot key for pyring: {}'.format(param_name))

    """set fixed parameters of pyring"""
    config_input = config_ini['input']
    mode = json.loads(config_input.get('inject-n-ds-modes', '{}'))
    fix_list_pyring = []
    for key in prior_parameters_dict.keys():
        if 'fix-' in key:
            param_name = key.replace('fix-', '')
            if param_name == 't':
                fix_list_pyring.append('t')
                for num in range(mode['t']):
                    param_name = f't_t_{num}'
                    fix_list_pyring.append(param_name)
            else:
                fix_list_pyring.append(param_name)
    if fix_list_pyring != []:
        for key in fix_list_pyring:
            if key in keys_to_plot_pyring:
                keys_to_plot_pyring.remove(key)
                print('not plot key : {}'.format(key))
    print('')

    """change parameterization"""
    if 'delta_f' in keys_to_plot_bilby:
        df_bilby['delta_f'] = df_bilby['delta_f'] + df_bilby['f1']
        df_bilby = df_bilby.rename(columns={'delta_f':'f2'})
        injection_parameters_dict_bilby['f2'] = injection_parameters_dict_bilby['delta_f'] + injection_parameters_dict_bilby['f1']
    if 'delta_tau' in keys_to_plot_bilby:
        df_bilby['delta_tau'] = df_bilby['delta_tau'] + df_bilby['tau1']
        df_bilby = df_bilby.rename(columns={'delta_tau':'tau2'})
        injection_parameters_dict_bilby['tau2'] = injection_parameters_dict_bilby['delta_tau'] + injection_parameters_dict_bilby['tau1']

    if bilby_label_v2 is not None:
        if 'delta_f' in keys_to_plot_bilby:
            df_bilby_v2['delta_f'] = df_bilby_v2['delta_f'] + df_bilby_v2['f1']
            df_bilby_v2 = df_bilby_v2.rename(columns={'delta_f':'f2'})
            injection_parameters_dict_bilby_v2['f2'] = injection_parameters_dict_bilby_v2['delta_f'] + injection_parameters_dict_bilby_v2['f1']
        if 'delta_tau' in keys_to_plot_bilby:
            df_bilby_v2['delta_tau'] = df_bilby_v2['delta_tau'] + df_bilby_v2['tau1']
            df_bilby_v2 = df_bilby_v2.rename(columns={'delta_tau':'tau2'})
            injection_parameters_dict_bilby_v2['tau2'] = injection_parameters_dict_bilby_v2['delta_tau'] + injection_parameters_dict_bilby_v2['tau1']
    
    """change log_A to A"""
    if 'logA_t_0' in df_pyring.columns:
        df_pyring['logA_t_0'] = 10**(df_pyring['logA_t_0']) * 1e19
        df_pyring = df_pyring.rename(columns={'logA_t_0':'A_t_0'})
    if 'logA_t_1' in df_pyring.columns:
        df_pyring['logA_t_1'] = 10**(df_pyring['logA_t_1']) * 1e19
        df_pyring = df_pyring.rename(columns={'logA_t_1':'A_t_1'})
    if 'logA' in df_pyring.columns:
        df_pyring['logA'] = 10**(df_pyring['logA']) * 1e17
        df_pyring = df_pyring.rename(columns={'logA':'A'})
    
    """set plot data"""
    posterior_bilby = [posterior_samples_bilby[key] for key in keys_to_plot_bilby]
    posterior_bilby = np.array(posterior_bilby).T
    use_df_bilby = pd.DataFrame(posterior_bilby, columns=keys_to_plot_bilby) # Create DataFrames for the posteriors
    use_df_bilby['source'] = 'bilby' # Add 'source' line
    if 'EPparam' in bilby_label:
        use_df_bilby['C'] = use_df_bilby['C'] * 1e21
        use_df_bilby['D'] = use_df_bilby['D'] * 1e17

    if bilby_label_v2 is not None:
        posterior_bilby_v2 = [posterior_samples_bilby_v2[key] for key in keys_to_plot_bilby]
        posterior_bilby_v2 = np.array(posterior_bilby_v2).T
        use_df_bilby_v2 = pd.DataFrame(posterior_bilby_v2, columns=keys_to_plot_bilby) # Create DataFrames for the posteriors
        use_df_bilby_v2['source'] = 'bilby v2' # Add 'source' line
        if 'EPparam' in bilby_label_v2:
            use_df_bilby_v2['C'] = use_df_bilby_v2['C'] * 1e21
            use_df_bilby_v2['D'] = use_df_bilby_v2['D'] * 1e17
        use_df_bilby = pd.concat([use_df_bilby, use_df_bilby_v2], ignore_index=True) # Combine bilby v1 and v2 data

    df_pyring['source'] = 'pyring' # Add 'source' line
    use_df_pyring = pd.DataFrame(columns=use_df_bilby.columns) # Create empty DataFrame with bilby columns
    for key in use_df_bilby.columns:
        if 'DSparam' in pyring_label:
            if key == 'A1':
                use_df_pyring[key] = df_pyring['A_t_0']
            elif key == 'A2':
                use_df_pyring[key] = df_pyring['A_t_1']
            elif key == 'f1':
                use_df_pyring[key] = df_pyring['f_t_0']
            elif key == 'f2':
                use_df_pyring[key] = df_pyring['f_t_1']
            elif key == 'tau1':
                use_df_pyring[key] = df_pyring['tau_t_0']
            elif key == 'tau2':
                use_df_pyring[key] = df_pyring['tau_t_1']
            elif key == 'phi1':
                use_df_pyring[key] = df_pyring['phi_t_0']
            elif key == 'phi2':
                use_df_pyring[key] = df_pyring['phi_t_1']
            elif key == 'source':
                use_df_pyring[key] = df_pyring['source']
        elif 'OTparam' in pyring_label or 'EPparam' in pyring_label:
            for key in use_df_bilby.columns:
                if key in df_pyring.columns:
                    use_df_pyring[key] = df_pyring[key]
                elif key == 'source':
                    use_df_pyring[key] = df_pyring['source']
        
        if 'EPparam' in pyring_label:
            use_df_pyring['C'] = use_df_pyring['C'] *1e21
            use_df_pyring['D'] = use_df_pyring['D'] *1e17

    use_data = pd.concat([use_df_bilby, use_df_pyring], ignore_index=True) # Combine bilby and pyring data


    """set latex format for error bar"""
    def format_error_latex_scaled(median, dif_lower, dif_upper, significant_digits=3):
        max_error = max(abs(dif_lower), abs(dif_upper))
        if max_error == 0:
            return f"${median}_{{-{dif_lower}}}^{{+{dif_upper}}}$"

        exponent = int(np.floor(np.log10(max_error)))

        if exponent <= -2:
            adjusted_exponent = int(np.floor(np.log10(abs(median))))
            scale = 10 ** adjusted_exponent

            median_scaled = median / scale
            dif_lower_scaled = abs(dif_lower) / scale
            dif_upper_scaled = abs(dif_upper) / scale

            # 有効数字に合わせて小数桁数を決定
            decimal_places = significant_digits - 1 - int(np.floor(np.log10(dif_upper_scaled)))
            decimal_places = max(decimal_places, 0)

            fmt_str = "{:." + str(decimal_places) + "f}"

            median_str = fmt_str.format(median_scaled)
            lower_str = fmt_str.format(dif_lower_scaled)
            upper_str = fmt_str.format(dif_upper_scaled)

            # 表示 (指数表記)
            error = rf"${median_str}_{{-{lower_str}}}^{{+{upper_str}}} \times 10^{{{adjusted_exponent}}}$"

        else:
            # exponent が -1 以上なら普通に表示
            decimal_places = significant_digits - 1 - int(np.floor(np.log10(abs(dif_upper))))
            decimal_places = max(decimal_places, 0)

            fmt_str = "{:." + str(decimal_places) + "f}"

            median_str = fmt_str.format(median)
            lower_str = fmt_str.format(abs(dif_lower))
            upper_str = fmt_str.format(abs(dif_upper))

            error = rf"${median_str}_{{-{lower_str}}}^{{+{upper_str}}}$"
        return error

    """Plot kdeplot for lower triangle"""
    def format_custom_f_g(median, dif_lower, dif_upper):
        max_error = max(abs(dif_lower), abs(dif_upper))
        exponent = int(np.floor(np.log10(max_error)))
        if exponent <= -3:
            fmt = '.2g'
            fmt = "{{0:{0}}}".format(fmt).format
            string_template = r"${{{0}}}_{{-{1}}}^{{+{2}}}$"
            error = string_template.format(fmt(median), fmt(dif_lower), fmt(dif_upper))
        else:
            fmt = '.3f'
            fmt = "{{0:{0}}}".format(fmt).format
            string_template = r"${{{0}}}_{{-{1}}}^{{+{2}}}$"
            error = string_template.format(fmt(median), fmt(abs(dif_lower)), fmt(abs(dif_upper)))
        return error
    
    def format_value(num, precision=3):
        if abs(num) < 0.1 and num != 0:
            return f"{num:.{precision + 2}f}"
        else:
            return f"{num:.{precision}f}"

    def kde_contour_plot(x, y, **kwargs):
        ax = plt.gca() #get current axes object
        if np.isnan(x).all() or np.isnan(y).all(): #nan value do not plotted
          return
        #sns.kdeplot(x=x, y=y, ax=ax, levels=[0.5, 1.0], fill=True, cut=0, alpha=0.9, **kwargs) #credible level 50
        #sns.kdeplot(x=x, y=y, ax=ax, levels=[0.1, 1.0], fill=True, cut=0, alpha=0.5, **kwargs) #credible level 90
        #sns.kdeplot(x=x, y=y, ax=ax, levels=[0.1, 0.5], cut=0, **kwargs)
    
        #The contour levels to draw. see https://corner.readthedocs.io/en/latest/api/#corner.hist2d(be careful the difference of "levels" in corner.py and seaborn)
        sns.kdeplot(x=x, y=y, ax=ax, levels=[np.exp(-9./2.), np.exp(-4./2.)], fill=True, cut=3, alpha=0.3, **kwargs) #3-sigma to 2-sigma
        sns.kdeplot(x=x, y=y, ax=ax, levels=[np.exp(-4./2.), np.exp(-1./2.)], fill=True, cut=3, alpha=0.6, **kwargs) #2-sigma to 1-sigma
        sns.kdeplot(x=x, y=y, ax=ax, levels=[np.exp(-1./2.), 1.0], fill=True, cut=3, alpha=0.9, **kwargs) #1-sigma to 0-sigma
        sns.kdeplot(x=x, y=y, ax=ax, levels=[np.exp(-9. / 2.), np.exp(-4. / 2.), np.exp(-1. / 2.)], cut=3, **kwargs) #contour line
    
    def custom_kdeplot(*args, **kwargs):
        ax = plt.gca()
        data = args[0]
    
        color = kwargs.pop("color") #recieve color index
        
        sns.kdeplot(data,
                    ax=ax, #set axes
                    fill=False, #fill color
                    color=color, #color of plot
                    cut=3, #cut off probability
                    warn_singular=False, 
                    **kwargs)
    
        """get quantiles"""
        quantiles = (0.05, 0.95)
        quants_to_compute = np.array([quantiles[0], 0.5, quantiles[1]])
        quants = np.percentile(data, quants_to_compute * 100)
        lower_limit = quants[0]
        median = quants[1]  
        upper_limit = quants[2]
        dif_upper = upper_limit - median 
        dif_lower = median - lower_limit
    
        """add quantiles line"""
        ax.axvline(lower_limit, color=color, linestyle='--', lw=1.5)
        ax.axvline(upper_limit, color=color, linestyle='--', lw=1.5)
        
        """set text of error bar"""
        # fmt = '.2f'
        # fmt = "{{0:{0}}}".format(fmt).format
        # string_template = r"${{{0}}}_{{-{1}}}^{{+{2}}}$"
        # error = string_template.format(fmt(median), fmt(dif_lower), fmt(dif_upper))
        #ax.set_title(error)

        # error = format_error_latex_scaled(median, dif_lower, dif_upper)

        error = format_custom_f_g(median, dif_lower, dif_upper)

        if color == (0.00392156862745098, 0.45098039215686275, 0.6980392156862745):
            offset = 0.15
            add_center_text(ax, error, 1.2 + offset, color)
        else:
            add_center_text(ax, error, 1.2, color)
        
        #y_min, y_max = ax.get_ylim()
        #ax.set_ylim(0, y_max*1.3)
    
    """histgram version"""
    def hist_2d_plot(x, y, **kwargs):
        ax = plt.gca()
        if np.isnan(x).all() or np.isnan(y).all():
            return
    
        color = kwargs.get("color", None)
    
        sns.histplot(x=x, y=y, ax=ax, 
                    bins=50, 
                    pthresh=0.05, 
                    cbar=False, 
                    color=color,
                    **kwargs)
    
    def fast_contour_plot(x, y, **kwargs):
        ax = plt.gca()
        color = kwargs.pop("color", "k")

        if isinstance(color, tuple):
            color = mcolors.to_hex(color)
        
        corner.hist2d(
                    x.values,
                    y.values,
                    ax=ax,
                    color=color, 
                    plot_datapoints=False,
                    plot_density=False, 
                    # levels=[1.0-np.exp(-0.5), 1.0-np.exp(-2.0), 1.0-np.exp(-4.5)]
                    levels=[0.5, 0.9],
                    )

    def custom_histplot(*args, **kwargs):
        ax = plt.gca()
        data = args[0]

        color = kwargs.pop("color") # colorを受け取る
    
        sns.histplot(data,
                    ax=ax,
                    element="step", 
                    fill=False,
                    stat="density",
                    color=color,
                    **kwargs)

        quantiles = (0.05, 0.95)
        quants_to_compute = np.array([quantiles[0], 0.5, quantiles[1]])
        quants = np.percentile(data, quants_to_compute * 100)
        lower_limit = quants[0]
        median = quants[1]  
        upper_limit = quants[2]
        dif_upper = upper_limit - median 
        dif_lower = median - lower_limit

        
        """add quantiles line"""
        ax.axvline(lower_limit, color=color, linestyle='--', lw=1.5)
        ax.axvline(upper_limit, color=color, linestyle='--', lw=1.5)
    
        """set text of error bar"""
        error = format_custom_f_g(median, dif_lower, dif_upper)
        
        color_index = palette.index(color)

        offset = 0.15 * (n_colors - color_index - 1)
        add_center_text(ax, error, 1.2 + offset, color)

        # if color == (0.00392156862745098, 0.45098039215686275, 0.6980392156862745):
        #     offset = 0.15
        #     add_center_text(ax, error, 1.2 + offset, color)
        # else:
        #     add_center_text(ax, error, 1.2, color)

    """add sub objects"""    
    def add_center_text(ax, text, hight, font_color):
        """add text to top of figure"""
        ax = ax
        x = 0.5
        y = hight
        ax.text(x, y, text, ha='center', va='center', fontsize=16, color=font_color, transform=ax.transAxes)
    
    def add_reference_lines_lower(x, y, **kwargs):
        ax = plt.gca()
        ref_values = kwargs.pop('ref_values') #get injection value
        x_ref, y_ref = ref_values.get(x.name, None), ref_values.get(y.name, None)
        """write injection value"""
        if x_ref is not None:
            ax.axvline(x=x_ref, color='red', linestyle=':', lw=2.3)
        if y_ref is not None:
            ax.axhline(y=y_ref, color='red', linestyle=':', lw=2.3)
    
    def add_reference_lines_diag(x, **kwargs):
        ax = plt.gca()
        color = kwargs.get('color', None)

        if n_colors > 1 and color is not None and color != palette[0]:
            return
        if getattr(ax, '_injection_ref_drawn', False):
            return

        ref_value = kwargs.pop('ref_value')
        x_ref = ref_value.get(x.name, None)
        print(x.name, x_ref)

        if x_ref is not None:
            ax.axvline(x=x_ref, color='red', linestyle=':', lw=2.3)
            
            x_min, x_max = ax.get_xlim()
            range_width = x_max - x_min
            
            buffer = range_width * 0.1 # 10% buffer
            
            if x_ref < x_min:
                ax.set_xlim(left=x_ref - buffer)
            elif x_ref > x_max:
                ax.set_xlim(right=x_ref + buffer)

            # value = r'${}$'.format(np.round(x_ref, 3))
            value = r'${}$'.format(format_value(x_ref, precision=3))

            add_center_text(ax, value, 1.07, 'red')
            ax._injection_ref_drawn = True
    
    """Make PairGrid"""
    n_colors = 2
    if bilby_label_v2 is not None:
        n_colors = 3

    sns.set_style("whitegrid")
    sns.set_context("paper")
    palette = sns.color_palette("colorblind", n_colors=n_colors)
    plt.style.use('~/research/my_plot_style.style')
    plot = sns.PairGrid(use_data, #data set
                        diag_sharey=False, #diagnal plot do not use different range
                        corner=True, #only oneside plot
                        hue='source', #use different color for each data
                        palette=palette, #color  
                        aspect=1.0)
    
    """add some plots"""
    if plot_kde:
        plot = plot.map_lower(kde_contour_plot)
        # plot = plot.map_lower(hist_2d_plot, alpha=0.3)
        plot.map_diag(custom_kdeplot, lw=2, alpha=0.7)
    else:
        plot = plot.map_lower(fast_contour_plot)
        plot.map_diag(custom_histplot, lw=2, alpha=0.7)

    plot.map_lower(add_reference_lines_lower, ref_values=injection_parameters_dict_bilby)
    plot.map_diag(add_reference_lines_diag, ref_value=injection_parameters_dict_bilby)

    """Add legend"""
    # handles = [Line2D([0], [0], color=palette[0], lw=4),
    #           Line2D([0], [0], color=palette[1], lw=4)]
    handles = [Line2D([0], [0], color=palette[i], lw=4) for i in range(n_colors)]
    
    # labels = [ "damped sinusoids", "overtone" ]
    # labels = ['bilby', 'pyring']
    labels = ['bilby_' + bilby_label, pyring_label]
    if bilby_label_v2 is not None:
        labels = ['bilby_' + bilby_label, 'bilby_' + bilby_label_v2, pyring_label]
    plot.fig.legend(
                    handles=handles,
                    labels=labels,
                    loc='upper right',
                    bbox_to_anchor=(0.98, 1.05),
                    fontsize=20
                    )
    
    """Adjust style""" 
    plot.fig.subplots_adjust(top=0.95)
    #plot.fig.suptitle('Your Title Here', fontsize=16)
    for ax in plot.axes.flatten(): #get latex label
        if ax is not None:
          label_x=ax.get_xlabel()
          label_y=ax.get_ylabel()
          ax.set_xlabel(latex_labels.get(label_x, label_x), fontsize=20)
          ax.set_ylabel(latex_labels.get(label_y, label_y), fontsize=20)
          ax.tick_params(axis='both', which='major', labelsize=14)
          ax.xaxis.get_offset_text().set_fontsize(12)
          ax.yaxis.get_offset_text().set_fontsize(12)
    
    """set file name of save fig"""
    def get_filename_without_extension(path):
        filename = os.path.basename(path)
        return os.path.splitext(filename)[0]
    
    if show_fig:
        plt.show()
        print(f'create posterior plot from: {path_json_bilby} and {path_posterior_pyring}')

    else:
        filename = f'{pyring_label}_bilby_{bilby_label}_comparison_posterior'
        if bilby_label_v2 is not None:
            filename = f'{pyring_label}_bilby_{bilby_label}_bilby_{bilby_label_v2}_comparison_posterior'
        save_filename = f'{filename}.pdf'
        plt.savefig(f'./{path_outdir}/{save_filename}', bbox_inches='tight', pad_inches=0.2, dpi=200)
        plt.close()
        print(f'create posterior plot {save_filename}')
        bilby.core.utils.logger.info(f'create posterior plot {save_filename}')


In [22]:
bilby_label_list = [
                    'shiftIm_to_220_dw0.1w1_snr100_DSparam_Mirror',
                    'shiftIm_to_220_dw0.01w1_snr100_DSparam_Mirror',
                    'shiftIm_to_220_dw0.001w1_snr100_DSparam_Mirror',
                    'shiftRe_to_220_dw0.1w1_snr100_DSparam_Mirror',
                    'shiftRe_to_220_dw0.01w1_snr100_DSparam_Mirror',
                    'shiftRe_to_220_dw0.001w1_snr100_DSparam_Mirror'
                    ]
bilby_label_list_v2 = [
                    'shiftIm_to_220_dw0.1w1_snr100_DSparam_Heaviside',
                    'shiftIm_to_220_dw0.01w1_snr100_DSparam_Heaviside',
                    'shiftIm_to_220_dw0.001w1_snr100_DSparam_Heaviside',
                    'shiftRe_to_220_dw0.1w1_snr100_DSparam_Heaviside',
                    'shiftRe_to_220_dw0.01w1_snr100_DSparam_Heaviside',
                    'shiftRe_to_220_dw0.001w1_snr100_DSparam_Heaviside'
                    ]
pyring_label_list = [
                     'pyring_shiftIm_to_220_dw0.1w1_snr100_DSparam',
                     'pyring_shiftIm_to_220_dw0.01w1_snr100_DSparam',
                     'pyring_shiftIm_to_220_dw0.001w1_snr100_DSparam',
                     'pyring_shiftRe_to_220_dw0.1w1_snr100_DSparam',
                     'pyring_shiftRe_to_220_dw0.01w1_snr100_DSparam',
                     'pyring_shiftRe_to_220_dw0.001w1_snr100_DSparam'
                      ]

for pyring_label, bilby_label, bilby_label_v2 in zip(pyring_label_list, bilby_label_list, bilby_label_list_v2):
    comparison_posterior_plot(pyring_label, bilby_label, default_plot_parameters=True, show_fig=False, plot_kde=False, bilby_label_v2=bilby_label_v2)

20:38 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:38 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object
20:38 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:38 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object


 injection_parameters bilby: {'A1': 0.292362984324464, 'A2': 0.292578588047657, 'f1': 201.23779332666524, 'f2': 201.23779332666524, 'tau1': 0.0033219623034593474, 'tau2': 0.0023201731795771175, 'phi1': 1.5707963267948966, 'phi2': -1.6091890156562867, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters bilby v2: {'A1': 0.292362984324464, 'A2': 0.292578588047657, 'f1': 201.23779332666524, 'f2': 201.23779332666524, 'tau1': 0.0033219623034593474, 'tau2': 0.0023201731795771175, 'phi1': 1.5707963267948966, 'phi2': -1.6091890156562867, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters pyring: {'A_t_0': 0.29236298432446395, 'A_t_1': 0.292578588047657, 'f_t_0': 201.23779332666524, 'f_t_1': 201.23779332666524, 'tau_t_0': 0.0033219623034593474, 'tau_t_1': 0.0023201

20:38 bilby INFO    : create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_DSparam_bilby_shiftIm_to_220_dw0.1w1_snr100_DSparam_Mirror_bilby_shiftIm_to_220_dw0.1w1_snr100_DSparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_DSparam_bilby_shiftIm_to_220_dw0.1w1_snr100_DSparam_Mirror_bilby_shiftIm_to_220_dw0.1w1_snr100_DSparam_Heaviside_comparison_posterior.pdf


20:39 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:39 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object
20:39 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:39 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object


...

create posterior plot pyring_shiftRe_to_220_dw0.01w1_snr100_DSparam_bilby_shiftRe_to_220_dw0.01w1_snr100_DSparam_Mirror_bilby_shiftRe_to_220_dw0.01w1_snr100_DSparam_Heaviside_comparison_posterior.pdf


20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_f2', 'name': 'f2', 'latex_label': '$f_2$ [Hz]', 'unit': None, 'boundary': None, 'minimum': 20, 'maximum': 500}, defaulting to base Prior object
20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_f2', 'name': 'f2', 'latex_label': '$f_2$ [Hz]', 'unit': None, 'boundary': None, 'minimum': 20, 'maximum': 500}, defaulting to base Prior object


 injection_parameters bilby: {'A1': 22.004161451791628, 'A2': 22.01261359425778, 'f1': 201.23779332666524, 'f2': 201.03093103007643, 'tau1': 0.0033219623034593474, 'tau2': 0.0033219623034593474, 'phi1': 0.0, 'phi2': 3.141592653589793, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters bilby v2: {'A1': 22.004161451791628, 'A2': 22.01261359425778, 'f1': 201.23779332666524, 'f2': 201.03093103007643, 'tau1': 0.0033219623034593474, 'tau2': 0.0033219623034593474, 'phi1': 0.0, 'phi2': 3.141592653589793, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters pyring: {'A_t_0': 22.004161451791624, 'A_t_1': 22.01261359425778, 'f_t_0': 201.23779332666524, 'f_t_1': 201.03093103007643, 'tau_t_0': 0.0033219623034593474, 'tau_t_1': 0.0033219623034593474, 'phi_t_0': 0.0, 'ph

20:40 bilby INFO    : create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_DSparam_bilby_shiftRe_to_220_dw0.001w1_snr100_DSparam_Mirror_bilby_shiftRe_to_220_dw0.001w1_snr100_DSparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_DSparam_bilby_shiftRe_to_220_dw0.001w1_snr100_DSparam_Mirror_bilby_shiftRe_to_220_dw0.001w1_snr100_DSparam_Heaviside_comparison_posterior.pdf


In [23]:
bilby_label_list = [
                    'shiftIm_to_220_dw0.1w1_snr100_OTparam_Mirror',
                    'shiftIm_to_220_dw0.01w1_snr100_OTparam_Mirror',
                    'shiftIm_to_220_dw0.001w1_snr100_OTparam_Mirror',
                    'shiftRe_to_220_dw0.1w1_snr100_OTparam_Mirror',
                    'shiftRe_to_220_dw0.01w1_snr100_OTparam_Mirror',
                    'shiftRe_to_220_dw0.001w1_snr100_OTparam_Mirror'
                    ]
bilby_label_list_v2 = [
                    'shiftIm_to_220_dw0.1w1_snr100_OTparam_Heaviside',
                    'shiftIm_to_220_dw0.01w1_snr100_OTparam_Heaviside',
                    'shiftIm_to_220_dw0.001w1_snr100_OTparam_Heaviside',
                    'shiftRe_to_220_dw0.1w1_snr100_OTparam_Heaviside',
                    'shiftRe_to_220_dw0.01w1_snr100_OTparam_Heaviside',
                    'shiftRe_to_220_dw0.001w1_snr100_OTparam_Heaviside'
                    ]
pyring_label_list = [
                     'pyring_shiftIm_to_220_dw0.1w1_snr100_OTparam',
                     'pyring_shiftIm_to_220_dw0.01w1_snr100_OTparam',
                     'pyring_shiftIm_to_220_dw0.001w1_snr100_OTparam',
                     'pyring_shiftRe_to_220_dw0.1w1_snr100_OTparam',
                     'pyring_shiftRe_to_220_dw0.01w1_snr100_OTparam',
                     'pyring_shiftRe_to_220_dw0.001w1_snr100_OTparam'
                      ]

for pyring_label, bilby_label, bilby_label_v2 in zip(pyring_label_list, bilby_label_list, bilby_label_list_v2):
    comparison_posterior_plot(pyring_label, bilby_label, default_plot_parameters=True, show_fig=False, plot_kde=False, bilby_label_v2=bilby_label_v2)

20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object
20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object


 injection_parameters bilby: {'A': 0.38, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.23779332666524, 'tau1': 0.0033219623034593474, 'tau2': 0.0023201731795771175, 'phiA': 0.0, 'phialpha': 0.0, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters bilby v2: {'A': 0.38, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.23779332666524, 'tau1': 0.0033219623034593474, 'tau2': 0.0023201731795771175, 'phiA': 0.0, 'phialpha': 0.0, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters pyring: {'A': 0.38, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.23779332666524, 'tau1': 0.0033219623034593474, 'tau2': 0.0023201731795771175, 'phiA': 0.0, 'phialpha': 0.0, 'psi': 0.0, 'ra': 0.0, 'dec': 0.0, 't': 0.0}

pl

20:40 bilby INFO    : create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_OTparam_bilby_shiftIm_to_220_dw0.1w1_snr100_OTparam_Mirror_bilby_shiftIm_to_220_dw0.1w1_snr100_OTparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_OTparam_bilby_shiftIm_to_220_dw0.1w1_snr100_OTparam_Mirror_bilby_shiftIm_to_220_dw0.1w1_snr100_OTparam_Heaviside_comparison_posterior.pdf


20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object
20:40 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:40 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_tau2', 'name': 'tau2', 'latex_label': '$\\tau_2$ [ms]', 'unit': None, 'boundary': None, 'minimum': 0.0005, 'maximum': 0.05}, defaulting to base Prior object


...

create posterior plot pyring_shiftRe_to_220_dw0.01w1_snr100_OTparam_bilby_shiftRe_to_220_dw0.01w1_snr100_OTparam_Mirror_bilby_shiftRe_to_220_dw0.01w1_snr100_OTparam_Heaviside_comparison_posterior.pdf


20:41 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:41 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_f2', 'name': 'f2', 'latex_label': '$f_2$ [Hz]', 'unit': None, 'boundary': None, 'minimum': 20, 'maximum': 500}, defaulting to base Prior object
20:41 bilby WARNING : Cannot load module priors_condition, returning function name as string
20:41 bilby WARNING : Unable to load prior <class 'bilby.core.prior.conditional.ConditionalUniform'> with arguments {'condition_func': 'priors_condition.condition_for_f2', 'name': 'f2', 'latex_label': '$f_2$ [Hz]', 'unit': None, 'boundary': None, 'minimum': 20, 'maximum': 500}, defaulting to base Prior object


 injection_parameters bilby: {'A': 0.28600000000000003, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.03093103007643, 'tau1': 0.0033219623034593474, 'tau2': 0.0033219623034593474, 'phiA': 0.0, 'phialpha': 0.0, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters bilby v2: {'A': 0.28600000000000003, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.03093103007643, 'tau1': 0.0033219623034593474, 'tau2': 0.0033219623034593474, 'phiA': 0.0, 'phialpha': 0.0, 'ra': 0.0, 'dec': 0.0, 'psi': 0.0, 'geocent_time': 0.0, 'fix_list': ['ra', 'dec', 'psi', 'geocent_time'], 'log_likelihood': nan, 'log_prior': nan}
 injection_parameters pyring: {'A': 0.286, 'alpha': 0.00029552945685847607, 'f1': 201.23779332666524, 'f2': 201.03093103007643, 'tau1': 0.0033219623034593474, 'tau2': 0.0033219623034593474, 'phiA': 0.0, 'phialpha': 0.0, 'psi': 0.0, 'ra':

20:41 bilby INFO    : create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_OTparam_bilby_shiftRe_to_220_dw0.001w1_snr100_OTparam_Mirror_bilby_shiftRe_to_220_dw0.001w1_snr100_OTparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_OTparam_bilby_shiftRe_to_220_dw0.001w1_snr100_OTparam_Mirror_bilby_shiftRe_to_220_dw0.001w1_snr100_OTparam_Heaviside_comparison_posterior.pdf


In [24]:
bilby_label_list = [
                    'shiftIm_to_220_dw0.1w1_snr100_EPparam_Heaviside',
                    'shiftIm_to_220_dw0.01w1_snr100_EPparam_Heaviside',
                    'shiftIm_to_220_dw0.001w1_snr100_EPparam_Heaviside',
                    'shiftRe_to_220_dw0.1w1_snr100_EPparam_Heaviside',
                    'shiftRe_to_220_dw0.01w1_snr100_EPparam_Heaviside',
                    'shiftRe_to_220_dw0.001w1_snr100_EPparam_Heaviside'
                    ]
pyring_label_list = [
                     'pyring_shiftIm_to_220_dw0.1w1_snr100_EPparam',
                     'pyring_shiftIm_to_220_dw0.01w1_snr100_EPparam',
                     'pyring_shiftIm_to_220_dw0.001w1_snr100_EPparam',
                     'pyring_shiftRe_to_220_dw0.1w1_snr100_EPparam',
                     'pyring_shiftRe_to_220_dw0.01w1_snr100_EPparam',
                     'pyring_shiftRe_to_220_dw0.001w1_snr100_EPparam'
                      ]

for pyring_label, bilby_label, bilby_label_v2 in zip(pyring_label_list, bilby_label_list, bilby_label_list_v2):
    comparison_posterior_plot(pyring_label, bilby_label, default_plot_parameters=True, show_fig=False, plot_kde=False, bilby_label_v2=None)

 injection_parameters bilby: {'C': 1.123011936062209, 'D': 0.38, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}
 injection_parameters pyring: {'C': 1.123011936062209, 'D': 0.38, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}

plot key for bilby: C
plot key for bilby: D
plot key for bilby: f
plot key for bilby: tau
plot key for bilby: phiC
plot key for bilby: phiD
not plot key for bilby: ra
not plot key for bilby: dec
not plot key for bilby: psi
not plot key for bilby: geocent_time

plot key for pyring: C
plot key for pyring: D
plot key for pyring: f
plot key for pyring: tau
plot key for pyring: phiC
plot key for pyring: phiD

C 1.123011936062209
C 1.123011936062209
D 0.38
D 0.38
f 201.23779332666524
f 201.23779332666524
tau 0.0033219623034593474
tau 0.0033219623034593474
phiC -3.141592653589793
phiC -3.141592653589793
phiD 0.0
phiD 0.0


20:41 bilby INFO    : create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_EPparam_bilby_shiftIm_to_220_dw0.1w1_snr100_EPparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftIm_to_220_dw0.1w1_snr100_EPparam_bilby_shiftIm_to_220_dw0.1w1_snr100_EPparam_Heaviside_comparison_posterior.pdf
 injection_parameters bilby: {'C': 0.8718118977325043, 'D': 0.295, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}
 injection_parameters pyring: {'C': 0.8718118977325043, 'D': 0.295, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}

plot key for bilby: C
plot key for bilby: D
plot key for bilby: f
plot key for bilby: tau
plot key for bilby: phiC
plot key for bilby: phiD
not plot key for bilby: ra
not plot key for bilby: dec
not plot key for bilby: psi
not plot key for bilby: geocent_time

plot key for pyring: C
plot key for pyring: D
plot key for pyring: f
plot key for pyring: tau
plot key for pyring: phiC
plot key for pyring: phiD

C 0.8718118977325043
C 0.8718118977325043
D 0.295
D 0.295
f 201.23779332666524
f 201.23779332666524
tau 0.003321962303

20:42 bilby INFO    : create posterior plot pyring_shiftIm_to_220_dw0.01w1_snr100_EPparam_bilby_shiftIm_to_220_dw0.01w1_snr100_EPparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftIm_to_220_dw0.01w1_snr100_EPparam_bilby_shiftIm_to_220_dw0.01w1_snr100_EPparam_Heaviside_comparison_posterior.pdf
 injection_parameters bilby: {'C': 0.8452142466152417, 'D': 0.28600000000000003, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}
 injection_parameters pyring: {'C': 0.8452142466152415, 'D': 0.286, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}

plot key for bilby: C
plot key for bilby: D
plot key for bilby: f
plot key for bilby: tau
plot key for bilby: phiC
plot key for bilby: phiD
not plot key for bilby: ra
not plot key for bilby: dec
not plot key for bilby: psi
not plot key for bilby: geocent_time

plot key for pyring: C
plot key for pyring: D
plot key for pyring: f
plot key for pyring: tau
plot key for pyring: phiC
plot key for pyring: phiD

C 0.8452142466152417
C 0.8452142466152417
D 0.28600000000000003
D 0.28600000000000003
f 201.2377933266

...

create posterior plot pyring_shiftRe_to_220_dw0.1w1_snr100_EPparam_bilby_shiftRe_to_220_dw0.1w1_snr100_EPparam_Heaviside_comparison_posterior.pdf
 injection_parameters bilby: {'C': 0.8422589520466569, 'D': 0.28500000000000003, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}
 injection_parameters pyring: {'C': 0.8422589520466567, 'D': 0.285, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}

plot key for bilby: C
plot key for bilby: D
plot key for bilby: f
plot key for bilby: tau
plot key for bilby: phiC
plot key for bilby: phiD
not plot key for bilby: ra
not plot key for bilby: dec
not plot key for bilby: psi
not plot key for bilby: geocent_time

plot key for pyring: C
plot key for pyring: D
plot key for pyring: f
plot key for pyring: tau
plot key for pyring: phiC
plot key for pyring: phiD

C 0.8422589520466569
C 0.8422589520466569
D 0.28500000000000003
D 0.28500000000000003
f 201.237793326665

20:42 bilby INFO    : create posterior plot pyring_shiftRe_to_220_dw0.01w1_snr100_EPparam_bilby_shiftRe_to_220_dw0.01w1_snr100_EPparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftRe_to_220_dw0.01w1_snr100_EPparam_bilby_shiftRe_to_220_dw0.01w1_snr100_EPparam_Heaviside_comparison_posterior.pdf
 injection_parameters bilby: {'C': 0.8452142466152417, 'D': 0.28600000000000003, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}
 injection_parameters pyring: {'C': 0.8452142466152415, 'D': 0.286, 'f': 201.23779332666524, 'tau': 0.0033219623034593474, 'phiC': -3.141592653589793, 'phiD': 0.0}

plot key for bilby: C
plot key for bilby: D
plot key for bilby: f
plot key for bilby: tau
plot key for bilby: phiC
plot key for bilby: phiD
not plot key for bilby: ra
not plot key for bilby: dec
not plot key for bilby: psi
not plot key for bilby: geocent_time

plot key for pyring: C
plot key for pyring: D
plot key for pyring: f
plot key for pyring: tau
plot key for pyring: phiC
plot key for pyring: phiD

C 0.8452142466152417
C 0.8452142466152417
D 0.28600000000000003
D 0.28600000000000003
f 201.2377933266

20:42 bilby INFO    : create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_EPparam_bilby_shiftRe_to_220_dw0.001w1_snr100_EPparam_Heaviside_comparison_posterior.pdf


create posterior plot pyring_shiftRe_to_220_dw0.001w1_snr100_EPparam_bilby_shiftRe_to_220_dw0.001w1_snr100_EPparam_Heaviside_comparison_posterior.pdf
